# CG2 Glaze Search on Google Colab (free GPU)

The same app as the Hugging Face Space [FBIWarninggg/glaze_search](https://huggingface.co/spaces/FBIWarninggg/glaze_search):
photo → ceramic objects → what the app found → glaze recipes, similar pottery, scores. Here it runs on Colab's GPU and
you get a public **gradio.live** link.

**How to use**
1. **Runtime → Change runtime type → T4 GPU → Save.**
2. **Runtime → Run all.**
3. Wait about 10–15 minutes (packages, then about 6 GB of data and models). Cell 5 prints the link.
4. Stop: **Runtime → Disconnect and delete runtime**.

Everything is downloaded from Hugging Face (public): the code from the Space, the data and models from the dataset
`FBIWarninggg/glaze-search-data`. The Space's own `app.py` and `fetch_data.py` run unchanged (on a normal GPU its ZeroGPU
parts do nothing). The link works while this notebook runs: Colab's free session ends after about 90 minutes without
activity or about 12 hours at most, and the link expires after 72 hours.

In [ ]:
# 1 · Check the GPU
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "no GPU")
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU > Save, then Runtime > Run all again"
print("torch", torch.__version__, "| CUDA", torch.version.cuda)

In [ ]:
# 2 · Download the Space's code (public)
from huggingface_hub import snapshot_download
APP = "/content/glaze_search"
snapshot_download("FBIWarninggg/glaze_search", repo_type="space", local_dir=APP)
print(open(f"{APP}/requirements.txt").read())

In [ ]:
# 3 · Install the Space's package versions. Colab's own torch / torchvision stay (they match Colab's GPU driver).
import re
req = [l.strip() for l in open(f"{APP}/requirements.txt") if l.strip() and not l.lstrip().startswith("#")]
req = [l for l in req if not re.match(r"(torch|torchvision)\b", l)]
open("/content/req_glaze.txt", "w").write("\n".join(req + ["gradio==6.29.1", "spaces"]) + "\n")
!pip install -q -r /content/req_glaze.txt

In [ ]:
# 4 · Data and models (about 6 GB): the Space's own start-up step
!cd {APP} && CG2_ROOT={APP} python fetch_data.py --part all

In [ ]:
# 5 · Start the app with a public link (Gradio share link, *.gradio.live)
import os, re, subprocess, sys, time
LOG = "/content/glaze_search.log"
env = dict(os.environ, GRADIO_SHARE="True", PYTHONUNBUFFERED="1", CG2_ROOT=APP, SKIP_FETCH="1")
proc = subprocess.Popen([sys.executable, "app.py"], cwd=APP, env=env, stdout=open(LOG, "w"), stderr=subprocess.STDOUT)
url, t0 = None, time.time()
while url is None:
    time.sleep(3)
    txt = open(LOG, errors="ignore").read()
    m = re.search(r"https://[a-z0-9-]+\.gradio\.live", txt)
    if m and "READY" in txt:
        url = m.group(0)
    elif proc.poll() is not None:
        print(txt[-4000:])
        raise SystemExit("The app stopped: see the end of its log above.")
    elif time.time() - t0 > 1800:
        print(txt[-4000:])
        raise SystemExit("No link after 30 minutes: see the log above.")
print(f"\nREADY after {time.time() - t0:.0f} s. Open (or share) this link:\n\n    {url}\n")

In [ ]:
# 6 · Keep the app running and show its log. The link works while this cell runs and the Colab session is open.
# Stop: press the ■ stop button of this cell, then Runtime > Disconnect and delete runtime.
import time
pos = 0
try:
    while proc.poll() is None:
        with open(LOG, errors="ignore") as f:
            f.seek(pos)
            new = f.read()
            pos = f.tell()
        if new:
            print(new, end="")
        time.sleep(5)
    print("\nThe app stopped (exit code", proc.returncode, ")")
except KeyboardInterrupt:
    proc.terminate()
    print("Stopped.")